# 07 Fig. 1

The collapse of the normalized summaries in `data/per_lane/Outputs`, written to `cone_mean_CI_outputs`. The line is the cross-network mean, and the band a 75% normal-theory prediction interval.


In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(os.path.join("..", "scripts")))
from paths import OUTPUTS_DIR, RESULTS

import os
import glob
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ========== CONFIG ==========
base_dir = OUTPUTS_DIR
pattern = os.path.join(base_dir, "outputs_*.txt")

out_dir = os.path.join(RESULTS, "cone_mean_CI_outputs")
os.makedirs(out_dir, exist_ok=True)

out_png = os.path.join(out_dir, "factors_cone_plot.png")
out_csv = os.path.join(out_dir, "factors_cone_curve.csv")

# Background layer mode: 'box' | 'iqr_points' | 'none'
BACKGROUND_MODE = 'none'

# Confidence level for mean CI / prediction PI
CONF_LEVEL = 0.75

# Interval mode: 'confidence' (mean CI) or 'prediction' (next-observation PI)
INTERVAL_MODE = 'prediction'

SHOW_MEAN_MARKERS = False
MEAN_MARKER_SIZE = 40

IQR_POINT_SIZE = 12
IQR_POINT_ALPHA = 0.25
BOX_WIDTH = 0.05
SHOW_FLIERS = False

# ========== HELPERS ==========
def safe_div(a, b):
    return a / b if (b not in [0, None] and not np.isnan(b)) else np.nan

def compute_fractions_direct(qc, kc, y3, y4, y5, y6, y7):
    """
    Fractions by your formulas:
      1/6: 1
      1/3: y4 / (2*y3)
      1/2: y5 / (3*y3)
      2/3: y6 / (4*y3)
      5/6: y7 / (5*y3)
      1  : y1 / (6*y3)  where y1 = qc
    """
    frac_1_6 = 1.0
    frac_1_3 = safe_div(y4, 2.0 * y3)
    frac_1_2 = safe_div(y5, 3.0 * y3)
    frac_2_3 = safe_div(y6, 4.0 * y3)
    frac_5_6 = safe_div(y7, 5.0 * y3)
    frac_1   = safe_div(qc, 6.0 * y3)
    return frac_1_6, frac_1_3, frac_1_2, frac_2_3, frac_5_6, frac_1

def ci_mean(arr, conf=0.95):
    """Mean ± critical * (s / sqrt(n)). Returns (mean, (lower, upper), n)."""
    a = np.asarray(arr, dtype=float)
    a = a[~np.isnan(a)]
    n = len(a)
    if n == 0:
        return np.nan, (np.nan, np.nan), 0
    mean_ = np.mean(a)
    std_  = np.std(a, ddof=1) if n > 1 else 0.0
    if n > 1 and std_ > 0:
        alpha = 1.0 - conf
        try:
            from scipy.stats import t
            crit = t.ppf(1 - alpha/2.0, df=n-1)
        except Exception:
            z_map = {0.80:1.2816, 0.90:1.6449, 0.95:1.96, 0.98:2.3263, 0.99:2.5758, 0.999:3.2905}
            crit = z_map.get(round(conf, 3), 1.96)
        margin = crit * std_ / math.sqrt(n)
    else:
        margin = 0.0
    return mean_, (mean_ - margin, mean_ + margin), n

def pi_next_obs(arr, conf=0.95):
    """Prediction interval for next observation: mean ± t*s*sqrt(1+1/n)."""
    a = np.asarray(arr, dtype=float)
    a = a[~np.isnan(a)]
    n = len(a)
    if n < 2:
        m = np.mean(a) if n == 1 else np.nan
        return m, m, n
    mean_ = np.mean(a)
    std_  = np.std(a, ddof=1)
    try:
        from scipy.stats import t
        tcrit = t.ppf(1 - (1-conf)/2.0, df=n-1)
    except Exception:
        z_map = {0.80:1.2816, 0.90:1.6449, 0.95:1.96, 0.98:2.3263, 0.99:2.5758, 0.999:3.2905}
        tcrit = z_map.get(round(conf, 3), 1.96)
    half = tcrit * std_ * math.sqrt(1.0 + 1.0/n)
    return mean_ - half, mean_ + half, n

# ========== 1) INGEST & COMPUTE FRACTIONS PER CITY ==========
records = []
for filepath in glob.glob(pattern):
    filename = os.path.basename(filepath)
    city_name = filename[len("outputs_"):-len(".txt")]

    with open(filepath, "r") as f:
        parts = [p.strip() for p in f.read().strip().split(",")]

    # Expect: [y1=qc, y2=kc, y3=q(kc/6), y4=q(kc/3), y5=q(kc/2), y6=q(2kc/3), y7=q(5kc/6)]
    if len(parts) != 7:
        y1, y2, y3, y4, y5, y6, y7 = [np.nan]*7
    else:
        y1, y2, y3, y4, y5, y6, y7 = map(float, parts)

    f_1_6, f_1_3, f_1_2, f_2_3, f_5_6, f_1 = compute_fractions_direct(
        qc=y1, kc=y2, y3=y3, y4=y4, y5=y5, y6=y6, y7=y7
    )

    records.append({
        "city": city_name,
        "k_over_kc_1_6": f_1_6,
        "k_over_kc_1_3": f_1_3,
        "k_over_kc_1_2": f_1_2,
        "k_over_kc_2_3": f_2_3,
        "k_over_kc_5_6": f_5_6,
        "k_over_kc_1":   f_1,
    })

df_plot = pd.DataFrame(records)

# ========== 2) SERIES & STATS ==========
x_points = np.array([0.0, 1/6, 1/3, 1/2, 2/3, 5/6, 1.0], dtype=float)

per_x_series = {
    1/6: df_plot["k_over_kc_1_6"].astype(float),
    1/3: df_plot["k_over_kc_1_3"].astype(float),
    1/2: df_plot["k_over_kc_1_2"].astype(float),
    2/3: df_plot["k_over_kc_2_3"].astype(float),
    5/6: df_plot["k_over_kc_5_6"].astype(float),
    1.0: df_plot["k_over_kc_1"].astype(float),
}

# We'll build the band and the MEAN-TO-PLOT.
# For prediction mode, the "mean" is redefined as midpoint of the (clamped) band.
means, lo_band, hi_band = [], [], []
for x in x_points:
    if x == 0.0:
        # At 0 we intentionally keep (1,1) and mean=1
        means.append(1.0); lo_band.append(1.0); hi_band.append(1.0)
    else:
        arr = per_x_series[x].values

        if INTERVAL_MODE == 'confidence':
            m, (lo, hi), _ = ci_mean(arr, conf=CONF_LEVEL)
            mean_to_plot = m  # standard mean for CI
        elif INTERVAL_MODE == 'prediction':
            lo, hi, _ = pi_next_obs(arr, conf=CONF_LEVEL)
            if hi is not None and not np.isnan(hi):
                hi = min(hi, 1.0)      # clamp upper end
            # plot the cross-network mean, as the caption says, not the
            # midpoint of the capped band
            mean_to_plot = np.nanmean(arr)
        else:
            raise ValueError("INTERVAL_MODE must be 'confidence' or 'prediction'")

        means.append(mean_to_plot); lo_band.append(lo); hi_band.append(hi)

means = np.array(means, dtype=float)
lo_band = np.array(lo_band, dtype=float)
hi_band = np.array(hi_band, dtype=float)

# ========== 3) SMOOTH (PCHIP if available; else linear) ==========
def make_smoother(x, y):
    try:
        from scipy.interpolate import PchipInterpolator
        return PchipInterpolator(x, y, extrapolate=False)
    except Exception:
        def _lin_interp(xq):
            return np.interp(xq, x, y)
        return _lin_interp

mean_smooth = make_smoother(x_points, means)
low_smooth  = make_smoother(x_points, lo_band)
high_smooth = make_smoother(x_points, hi_band)

xx = np.linspace(0.0, 1.0, 400)
yy_mean = mean_smooth(xx) if callable(getattr(mean_smooth, "__call__", None)) else mean_smooth(xx)
yy_low  = low_smooth(xx)  if callable(getattr(low_smooth, "__call__", None))  else low_smooth(xx)
yy_high = high_smooth(xx) if callable(getattr(high_smooth, "__call__", None)) else high_smooth(xx)

# Clamps
yy_low  = np.clip(yy_low,  0, None)
yy_high = np.clip(yy_high, 0, None)
yy_mean = np.clip(yy_mean, 0, None)

# If in prediction mode, enforce the upper clamp on the smoothed envelope too
if INTERVAL_MODE == 'prediction':
    yy_high = np.minimum(yy_high, 1.0)
    # no re-centring; the line stays the smoothed mean

# ========== 4) PLOT ==========
plt.figure(figsize=(8, 5), dpi=140)

# Background layer
x_for_bg = [1/3, 1/2, 2/3, 5/6, 1.0]
data_for_bg = [per_x_series[x].dropna().values for x in x_for_bg]

if BACKGROUND_MODE == 'box':
    plt.boxplot(
        data_for_bg,
        positions=x_for_bg,
        widths=BOX_WIDTH,
        showfliers=SHOW_FLIERS
    )
elif BACKGROUND_MODE == 'iqr_points':
    for x, arr in zip(x_for_bg, data_for_bg):
        if arr.size == 0:
            continue
        q1, q3 = np.percentile(arr, [25, 75])
        inside = arr[(arr >= q1) & (arr <= q3)]
        plt.scatter(np.full_like(inside, x, dtype=float), inside,
                    s=IQR_POINT_SIZE, alpha=IQR_POINT_ALPHA)
elif BACKGROUND_MODE == 'none':
    pass
else:
    raise ValueError("Invalid BACKGROUND_MODE. Use 'box', 'iqr_points', or 'none'.")

# Band + "mean"
plt.fill_between(xx, yy_low, yy_high, alpha=0.25, linewidth=0)
plt.plot(xx, yy_mean, linewidth=2)

# Optional discrete mean markers
if SHOW_MEAN_MARKERS:
    plt.scatter(x_points, means, s=MEAN_MARKER_SIZE)

plt.xlabel("K / Kc")
plt.ylabel("Fraction to scale (density * uf)")
title_interval = "Mean ± CI" if INTERVAL_MODE == 'confidence' else "Mean, PI upper end capped at 1"
plt.title(f"Scaling Fraction vs. Normalized Density ({title_interval}, {int(CONF_LEVEL*100)}%)")

# y-limits
ymax_candidates = [np.nanmax(a) for a in data_for_bg if a.size > 0] if BACKGROUND_MODE != 'none' else []
ymax = np.nanmax([np.nanmax(yy_high)] + ymax_candidates) if (ymax_candidates or np.isfinite(np.nanmax(yy_high))) else np.nanmax(yy_high)
if np.isfinite(ymax):
    plt.ylim(0, ymax * 1.05)

plt.xlim(0, 1.0)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(out_png, bbox_inches="tight")
plt.show()

# ========== 5) SAVE SMOOTHED CURVES ==========
curve_df = pd.DataFrame({
    "k_over_kc": xx,
    "mean": yy_mean,
    "band_lower": yy_low,
    "band_upper": yy_high,
    "conf_or_pred": INTERVAL_MODE,
    "level": CONF_LEVEL
})
curve_df.to_csv(out_csv, index=False)

print("Saved plot to:", out_png)
print("Saved curve CSV to:", out_csv)
print(f"Mode: {INTERVAL_MODE}, Level: {CONF_LEVEL:.3f}")
